# Tabla: astronomy_daily_silver
📌 **Descripción**  
Tabla a nivel diario que contiene información astronómica (amanecer, atardecer, fases lunares) por ciudad y fecha.

Cada registro representa:

- 1 ciudad
- 1 día (fecha del registro)
- Últimos datos disponibles (según `ingestion_time`)

📊 **Granularidad**  
`city + date`

🧱 **Columnas**

| Columna           | Tipo       | Descripción |
|------------------|-----------|------------|
| city             | string    | Nombre de la ciudad (proveniente de metadata) |
| date             | date      | Fecha del registro |
| ingestion_time   | timestamp | Momento en que se ingirieron los datos |
| sunrise          | string    | Hora de salida del Sol (formato AM/PM) |
| sunset           | string    | Hora de puesta del Sol (formato AM/PM) |
| sunrise_ts       | timestamp | Hora de salida del Sol en formato timestamp |
| sunset_ts        | timestamp | Hora de puesta del Sol en formato timestamp |
| moonrise         | string    | Hora de salida de la Luna (formato AM/PM) |
| moonset          | string    | Hora de puesta de la Luna (formato AM/PM) |
| moon_phase       | string    | Fase lunar (ej. Waxing Gibbous) |
| moon_illumination| long      | Iluminación lunar (%) |

⚙️ **Lógica aplicada**

- Se seleccionan datos desde: `data.astronomy.astro`  
- Se genera un nivel intermedio: 1 fila = 1 día por ciudad  
- Se construyen timestamps `sunrise_ts` y `sunset_ts` combinando `date + sunrise/sunset`  
- Se eliminan duplicados utilizando ventana:

```python
partitionBy(city, date)
orderBy(ingestion_time DESC)


In [0]:
from pyspark.sql.functions import col,explode,lower, regexp_replace,desc,row_number,to_timestamp,to_date,hour,concat_ws
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
df_astronomy_bronze = spark.read.json("/Volumes/workspace/default/bronce_clima/astronomy/")
df_astronomy_bronze.show(5)
df_astronomy_bronze.printSchema()

In [0]:
# 1. Selección de columnas correctas
df_astronomy = df_astronomy_bronze.select(
    col("city"),
    col("date"),
    col("data.astronomy.astro.sunrise").alias("sunrise"),
    col("data.astronomy.astro.sunset").alias("sunset"),
    col("data.astronomy.astro.moonrise").alias("moonrise"),
    col("data.astronomy.astro.moonset").alias("moonset"),
    col("data.astronomy.astro.moon_phase").alias("moon_phase"),
    col("data.astronomy.astro.moon_illumination").alias("moon_illumination"),
    col("metadata.ingestion_time").alias("ingestion_time")
)

# 2. Convertir ingestion_time
df_astronomy = df_astronomy.withColumn(
    "ingestion_time",
    to_timestamp("ingestion_time", "yyyy-MM-dd'T'HH-mm-ss'Z'")
)

# 3. Eliminar duplicados (quedarte con el más reciente)
window_spec = Window.partitionBy("city", "date").orderBy(desc("ingestion_time"))

df_astronomy_latest = (
    df_astronomy
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
df_astronomy_latest = df_astronomy_latest \
    .withColumn("sunrise_ts", to_timestamp(concat_ws(" ", col("date"), col("sunrise")), "yyyy-MM-dd hh:mm a")) \
    .withColumn("sunset_ts", to_timestamp(concat_ws(" ", col("date"), col("sunset")), "yyyy-MM-dd hh:mm a"))
df_astronomy_latest = df_astronomy_latest.drop("sunrise", "sunset")
df_astronomy_latest = df_astronomy_latest.drop("sunrise", "sunset")


In [0]:
df_astronomy_latest.show()
df_astronomy_latest.printSchema()

In [0]:
df_astronomy_latest.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_astronomy_daily_silver")

In [0]:
spark.table("weather_astronomy_daily_silver").show(5)